In [2]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

from pymilvus.model.hybrid import BGEM3EmbeddingFunction

In [3]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('.', 'NIR')))

In [4]:
from parser_2 import Parser2

In [5]:
xml_file_path = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr34.xml'

data = Parser2.XLMtoString(xml_file_path)

In [6]:
display(data[:10])

only_text = [x['text'] for x in data]

display(only_text[:10])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195',
  'text': 'A Fishmo

['Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660',
 'The Annunciation by Francesco Solimena, dated 1693 - 1693',
 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611',
 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579',
 'St Barbara by Parmigianino, dated None',
 "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737",
 'A Fishmonger at the Door by Jacob Ochtervelt, dated 1663 - 1663',
 'Portrait of a Deceased Girl, probably Catharina Margaretha van Valkenburg by Johannes Thopas, dated 1682 - 1682',
 'View of the Dam and the Damrak in Amsterdam by Jacob van Ruisdael, dated 1675 - 1672',
 'Hall Settle by Anonymous (Northern Netherlands), dated 1720 - 1700']

In [7]:
# Generate embeddings using BGEM3 model
ef = BGEM3EmbeddingFunction(use_fp16=False, device="cpu")
embeddings = ef(only_text)
# Debug prints to verify the dimensions
print(f"Number of texts: {len(only_text)}")
print(f"Dense embeddings shape: {len(embeddings['dense'])}")
print(f"Sparse embeddings shape: {(embeddings['sparse']).shape}")

# Prepare data for insertion
entities = [
    [item['id'] for item in data],  # IDs
    only_text,  # Texts
    embeddings["dense"],  # Dense vectors
    embeddings["sparse"]  # Sparse vectors
]

# Verify the lengths of each component to ensure they match
print(f"Length of IDs: {len(entities[0])}")
print(f"Length of texts: {len(entities[1])}")
print(f"Shape of dense vectors: {len(entities[2])}")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 52/52 [00:40<00:00,  1.30it/s]

Number of texts: 831
Dense embeddings shape: 831
Sparse embeddings shape: (831, 250002)
Length of IDs: 831
Length of texts: 831
Shape of dense vectors: 831


In [12]:
display(entities[2][0])

array([-0.0239582 ,  0.01874152, -0.02132395, ..., -0.01544944,
       -0.05640083, -0.02763552], dtype=float32)

In [13]:
entities[2][0].shape

type(entities[2][0])

numpy.ndarray

In [14]:
from pymilvus import MilvusClient
from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

# client = MilvusClient("milvus_demo.db")


In [15]:
connections.connect("default", host="localhost", port="19530")

In [16]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=1024),  # Ensure the dimension matches your embeddings
    # Store sparse vectors
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),  
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'hybrid_demo'
col = Collection(col_name, schema, consistency_level="Strong")

In [17]:
sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

In [305]:
# if col.has_collection(col_name):
#     print("Collection exists.")
    
# # # Create an index for the dense vector field
# # dense_index = client.prepare_index_params(index_type="IVF_FLAT", metric_type="L2", params={"nlist": 1024})

# # client.create_index(col_name, dense_index)

In [ ]:
# sparse_index = client.prepare_index_params(index_type="FLAT", metric_type="JACCARD", params={"nlist": 1024})
# client.create_index("sparse_vector", sparse_index)

In [18]:
col.insert(entities)

(insert count: 831, delete count: 0, upsert count: 0, timestamp: 451820094746787842, success count: 831, err count: 0, cost: 0)

In [19]:
col.flush()

In [23]:
query = "Johannes Vermeer"
query_embeddings = ef([query])
k=10
# Prepare the search requests for both vector fields
sparse_search_params = {"metric_type": "IP"}
sparse_req = AnnSearchRequest(query_embeddings["sparse"],
                              "sparse_vector", sparse_search_params, limit=k)
dense_search_params = {"metric_type": "IP"}
dense_req = AnnSearchRequest(query_embeddings["dense"],
                             "dense_vector", dense_search_params, limit=k)

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.hybrid_search([sparse_req, dense_req], rerank=RRFRanker(),
                        limit=k, output_fields=['text'])

for result in res[0]:
    print(result)

id: ID: /2021672/resource_document_mauritshuis_406, distance: 0.032522473484277725, entity: {'text': 'Diana and her Nymphs by Johannes Vermeer, dated 1654 - 1653'}
id: ID: /2021672/resource_document_mauritshuis_92, distance: 0.03226646035909653, entity: {'text': 'View of Delft by Johannes Vermeer, dated 1661 - 1660'}
id: ID: /2021672/resource_document_mauritshuis_670, distance: 0.0320020467042923, entity: {'text': 'Girl with a Pearl Earring by Johannes Vermeer, dated 1665 - 1665'}
id: ID: /2021672/resource_document_mauritshuis_809, distance: 0.03125, entity: {'text': 'A Farmstead by the Dunes by Jan Vermeer van Haarlem, dated 1648 - 1648'}
id: ID: /2021672/resource_document_mauritshuis_724, distance: 0.03076923079788685, entity: {'text': 'Landscape on the Edge of the Dunes by Jan Vermeer van Haarlem, dated 1648 - 1648'}
id: ID: /2021672/resource_document_mauritshuis_662, distance: 0.029857397079467773, entity: {'text': 'Still Life with Books, a Globe and Musical Instruments by Jan Verm